<a href="https://colab.research.google.com/github/oyatillonewuu/dblabs/blob/main/lab12/solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("IP Counter") \
       .getOrCreate()

spark

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Ign:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Ign:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Ign:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Err:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
  Could not connect to security.ubuntu.com:80 (91.189.92.22), connection timed out Could not connect to security.ubuntu.com:80 (185.125.190.81), connection timed out Could not connect to security.ubuntu.com:80 (185.125.190.83), connection timed out Could not connect to security.ubuntu.com:80 (91.189.91.83), connection timed out Could not connect to security.ubuntu.com:80 (91.189.91.81)

In [14]:
# Initialize file paths.

logs_path = "/content/server_logs.log"
output_path = "/content/ip_counts.json"

In [15]:
# Load the records from the file to spark.


logs_rdd = spark.sparkContext.textFile(logs_path, minPartitions=4)

In [16]:
# Apply map which emits (ip, 1).
# In a log record:
#   log.substring.until( index_of( first_space ) ) = ip
# The function for map is based on this observation.

ips_rdd = logs_rdd.map(
    lambda log: (
      log[:log.index(" ")], 1
    )
)

In [17]:
# Apply reduce: shuffles by keys into partitions & combines items by their key.

ip_counts_rdd = ips_rdd.reduceByKey(lambda k1, k2: k1 + k2)

In [18]:
# Collect the final result.

ip_counts = ip_counts_rdd.collect()

In [19]:
# Create a dict form of the result.

ip_counts_dict = {
    ip: count for ip, count in ip_counts
}

In [20]:
# Save to a 'json' file.

import json

with open(output_path, "w") as output_file:
  json.dump(ip_counts_dict, output_file, indent=4)